<a href="https://colab.research.google.com/github/enm0910/ST554/blob/main/EMartinez_Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Emma Martinez

Course: ST 554 Project 3

Purpose: Project 3- PySpark ML + Streaming Project

# Project #3

This notebook has two parts:

**Part 1** – Fits an elastic net regression model to predict Power_Zone_3 (power consumption in Tetouan City) using PySpark's MLlib. We load the data, build a transformation pipeline (binarization, one-hot encoding, PCA), and tune the model via 5-fold cross-validation across a grid of regularization parameters.

**Part 2** – Reads a live stream of incoming CSV files using PySpark Structured Streaming. We apply the fitted model to each batch of arriving data to generate real-time predictions, compute residuals, and write the results to the console.

We start every section by installing PySpark and creating (or reusing) a SparkSession.

### Setup

In [3]:
# Install PySpark for every session
!pip install pyspark

In [4]:
# Imports
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField,
                                DoubleType, IntegerType, StringType)
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Binarizer, OneHotEncoder, StringIndexer,
    VectorAssembler, PCA, SQLTransformer
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [6]:
# Start a local Spark session
spark = SparkSession.builder \
    .appName("PowerZone3Prediction") \
    .getOrCreate()

# Surpress normal Spark logs
spark.sparkContext.setLogLevel("ERROR")
print("Spark session started")

Spark session started


### Part 1: Model Fitting